In [27]:
import duckdb
import os

In [14]:
# BASELINE MODEL: predict target=last item in the input sequence
query_baseline_model = """
SELECT AVG(CASE WHEN input_sequence[-1] == target_item THEN 1 ELSE 0 END) as accuracy_pred_last_item
FROM '../data/features/sequence_chunks/*.parquet'
"""
print("Accuracy of predicting the last item in the input sequence:\n", duckdb.query(query_baseline_model).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Accuracy of predicting the last item in the input sequence:
    accuracy_pred_last_item
0                 0.485467


In [25]:
# MODEL 1: Markov Chain

# Create output_path to save our markov_model
os.makedirs('../data/models', exist_ok=True)

# Transition Matrix for The Markov Chain
query_transition_matrix = """
COPY (WITH ranked_pairs AS (
    SELECT 
        input_sequence[-1] as last_item, 
        target_item, 
        COUNT(*) as count,
        ROW_NUMBER() OVER (PARTITION BY input_sequence[-1] ORDER BY count DESC) as rn_pairs
    FROM '../data/features/sequence_chunks/*.parquet'
    GROUP BY last_item, target_item)
SELECT target_item, last_item
FROM ranked_pairs WHERE rn_pairs=1) TO '../data/models/markov_model.parquet' (FORMAT parquet);
"""

# Run the query
duckdb.query(query_transition_matrix)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [26]:
# Calculate the accuracy of markov_model
query_accuracy_markov_model = """ 
    SELECT AVG(CASE WHEN s.target_item = m.target_item THEN 1 ELSE 0 END) as accuracy
    FROM  '../data/features/sequence_chunks/*.parquet' as s
    LEFT JOIN '../data/models/markov_model.parquet' as m
    ON s.input_sequence[-1] = m.last_item
"""

# Run the score
print("Accuracy of markov_model:\n", duckdb.query(query_accuracy_markov_model).df())

Accuracy of markov_model:
    accuracy
0  0.490439
